# Text Preprocessing
Step-by-step NLP preprocessing pipeline for Steam game reviews.

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import string

In [2]:
import nltk

nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /home/ritesh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words("english"))

In [4]:
import spacy

## 2. Load Cleaned Dataset

In [5]:
df = pd.read_csv("../data/processed/cleaned_reviews.csv")
df.head()

,recommendationid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,...,created,updated,steamid,num_games_owned,num_reviews,playtime_forever,playtime_last_two_weeks,playtime_at_review,last_played,review_length
0,153486817,english,It's very fun. I don't usually like open world...,1702441215,1702441215,True,0,0,0.0,0,...,2023-12-13,2023-12-13,76561198019963558,0,8,2482.0,2482.0,2452,1.702444e+09,209
1,153486487,english,fun,1702440650,1702440650,True,0,0,0.0,0,...,2023-12-13,2023-12-13,76561199103884291,0,9,6634.0,441.0,6622,1.702441e+09,3
2,153486273,english,Fun game,1702440259,1702440259,True,0,0,0.0,0,...,2023-12-13,2023-12-13,76561198131623718,354,3,10328.0,3.0,10324,1.702440e+09,8
3,153486133,english,cyberpunk,1702440019,1702440019,True,0,0,0.0,0,...,2023-12-13,2023-12-13,76561198273553741,0,5,1649.0,1649.0,1580,1.702444e+09,9
4,153485556,english,Coming back to try the game after 2.0 came out...,1702439168,1702439168,False,0,0,0.0,0,...,2023-12-13,2023-12-13,76561198364138475,86,8,4070.0,9.0,4070,1.702439e+09,428


## 3. Sample Review
Pick the first review as a working sample to walk through each preprocessing step.

In [6]:
sample = df.loc[0, "review"]

print(sample)

It's very fun. I don't usually like open world games, but this one is very nice. Writing, plot, characters are all engaging and the gameplay is fluid and open-ended. It reminds me a bit of Deus Ex, but better.


### 3a. Lowercase

In [7]:
sample = sample.lower()

### 3b. Remove URLs

In [8]:
sample = re.sub(r"http\S+|www\S+", "", sample)
print(sample)

it's very fun. i don't usually like open world games, but this one is very nice. writing, plot, characters are all engaging and the gameplay is fluid and open-ended. it reminds me a bit of deus ex, but better.


### 3c. Remove HTML

In [9]:
re.sub(r"<.*?>", "", sample)

"it's very fun. i don't usually like open world games, but this one is very nice. writing, plot, characters are all engaging and the gameplay is fluid and open-ended. it reminds me a bit of deus ex, but better."

### 3d. Remove Punctuation

In [10]:
sample = sample.translate(
    str.maketrans("", "", string.punctuation)
)

### 3e. Remove Numbers

In [11]:
sample = re.sub(r"\d+", "", sample)
print(sample)

its very fun i dont usually like open world games but this one is very nice writing plot characters are all engaging and the gameplay is fluid and openended it reminds me a bit of deus ex but better


### 3f. Tokenization

In [12]:
tokens = sample.split()

print(tokens)

['its', 'very', 'fun', 'i', 'dont', 'usually', 'like', 'open', 'world', 'games', 'but', 'this', 'one', 'is', 'very', 'nice', 'writing', 'plot', 'characters', 'are', 'all', 'engaging', 'and', 'the', 'gameplay', 'is', 'fluid', 'and', 'openended', 'it', 'reminds', 'me', 'a', 'bit', 'of', 'deus', 'ex', 'but', 'better']


In [13]:
type(tokens)

list

### 3g. Stopword Removal

In [14]:
filtered_tokens = [
    word
    for word in tokens
    if word not in stop_words
]

print(filtered_tokens)

['fun', 'dont', 'usually', 'like', 'open', 'world', 'games', 'one', 'nice', 'writing', 'plot', 'characters', 'engaging', 'gameplay', 'fluid', 'openended', 'reminds', 'bit', 'deus', 'ex', 'better']


### 3h. Lemmatization

In [16]:
import spacy

nlp = spacy.load("en_core_web_sm")
doc = nlp(" ".join(filtered_tokens))

lemmatized_tokens = [
    token.lemma_
    for token in doc
]

print(lemmatized_tokens)

['fun', 'do', 'not', 'usually', 'like', 'open', 'world', 'game', 'one', 'nice', 'writing', 'plot', 'character', 'engage', 'gameplay', 'fluid', 'openended', 'remind', 'bit', 'deus', 'ex', 'well']


### 4.preprocess() function

In [35]:
import re
import string
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def preprocess(text):
    # Handle missing values
    if pd.isna(text):
        return ""

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Tokenize + Remove stopwords + Keep only alphabetic words
    tokens = [
        word
        for word in text.split()
        if word not in stop_words and word.isalpha()
    ]

    # Join back into a sentence
    return " ".join(tokens)

In [ ]:
df["clean_review"] = df["review"].apply(preprocess)

,review,clean_review
0,It's very fun. I don't usually like open world...,fun dont usually like open world games one nic...
1,fun,fun
2,Fun game,fun game
3,cyberpunk,cyberpunk
4,Coming back to try the game after 2.0 came out...,coming back try game came however game wont lo...


In [40]:
df[["review", "clean_review"]].head(20)

,review,clean_review
0,It's very fun. I don't usually like open world...,fun dont usually like open world games one nic...
1,fun,fun
2,Fun game,fun game
3,cyberpunk,cyberpunk
4,Coming back to try the game after 2.0 came out...,coming back try game came however game wont lo...
5,i dont even own this fucking game why can i wr...,dont even fucking game write review
6,Todo valio la pena al final con el mejor endin...,todo valio la pena al final con el mejor endin...
7,I am a sneaky boi and I stab people with arm s...,sneaky boi stab people arm swords
8,Should have brought it when the game was in sc...,brought game scrambles worth
9,"The first two hours is already fun for me, but...",first two hours already fun game grilling comp...


In [37]:
df = df[df["clean_review"].str.strip() != ""]

In [41]:
df.to_csv("../data/processed/preprocessed_reviews.csv", index=False)